# Milestone 1

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [ ]:
Basic inspect

In [2]:
df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

df.columns.tolist()


['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']

some more required dependencies

In [3]:
import string
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## Q1. Sum of most frequent and least frequest correct options

In [4]:
freq = df['answer'].value_counts()
freq

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

In [5]:
q1_answer = freq.max() + freq.min()
q1_answer

Q1 result: 814


## Q2. Total unique words in prompt column

In [7]:
def clean_text(text):
    text = str(text).lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return text
    
unique_words_in_prompt = set()

for txt in df['prompt'].fillna(""):
    cleaned = clean_text(txt)
    unique_words_in_prompt.update(cleaned.split())

len(unique_words_in_prompt)

Q2 vocabulary size: 859


## Q3. Meaningful words in [row1, prompt] cell 
Here meaningful words mean words excluding ENGLISH_STOP_WORDS like the, is, a, etc.

In [8]:
# Q3
row1 = clean_text(df.loc[0, 'prompt'])
tokens = row1.split()
filtered = [w for w in tokens if w not in ENGLISH_STOP_WORDS]
filtered

['pick',
 'best',
 'possible',
 'answer',
 'martin',
 'heideggers',
 'view',
 'relationship',
 'time',
 'human',
 'existence',
 'listed',
 'options']

In [9]:
len(filtered)

13

In [10]:
# Ensuring there is no missing values
df.isna().sum()

id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

## Q4. Vocabulary size generated by vectorizer

In [11]:
target_cols = ['prompt', 'A', 'B', 'C', 'D', 'E']

all_text = df[target_cols].astype(str).agg(' '.join, axis=1).tolist()

vectorizer = TfidfVectorizer(stop_words="english")
vectorizer.fit(all_text)

vectorizer.get_feature_names_out()

array(['000', '013', '013343', ..., 'zweig', 'über', 'φ0'],
      shape=(2762,), dtype=object)

In [12]:
len(vectorizer.get_feature_names_out())

2762

## Q5. Consine similarity for [row1, prompt] and [row1, A]

In [13]:
prompt_vec = vectorizer.transform([str(df.loc[0, 'prompt'])])
option_a_vec = vectorizer.transform([str(df.loc[0, 'A'])])

score = cosine_similarity(prompt_vec, option_a_vec)[0,0]

round(score,4)

np.float64(0.272)

## Q6. Cosine similarity between the prompt and each of its 5 options . Then the percentage of instances where the option with the highest cosine similarity matches the correct answer. 

In [14]:
options = ['A', 'B', 'C', 'D', 'E']

prompt_matrices = vectorizer.transform(df['prompt'].astype(str))

sim_scores = {}
for opt in options:
    opt_matrices = vectorizer.transform(df[opt].astype(str))
    
    sim_scores[opt] = np.array(prompt_matrices.multiply(opt_matrices).sum(axis=1)).flatten()

matrix_scores = np.column_stack([sim_scores[opt] for opt in options])

best_indices = np.argmax(matrix_scores, axis=1)

predicted_letters = np.array(options)[best_indices]

accuracy = 100 * np.mean(predicted_letters == df['answer'])

accuracy

np.float64(13.15)

## Q7 and Q8
**MAP@3:** The score is the reciprocal of the rank of the correct answer within the top 3 predictions; otherwise, it is 0.

- Ground truth = C, Prediction = C A B → MAP@3 = 1/1 = **1.0**
- Ground truth = B, Prediction = D B E → MAP@3 = 1/2 = **0.5**

## Q9. Overall MAP@3 score of *Majority Class* baseline
The Majority Class Baseline: Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall MAP@3 score of this "Majority Class" baseline? 

In [15]:
ranking = list(freq.index[:3]) # freq from Q1

score_map = {
    ranking[0]: 1.0,
    ranking[1]: 0.5,
    ranking[2]: 1/3
}

map3 = df['answer'].map(score_map).fillna(0)

map3.mean()

np.float64(0.42125)

## Q10. Average MAP@3 score
The TF-IDF Pipeline: Build a basic pipeline that evaluates every row in train.csv. For each row, calculate the TF-IDF cosine similarity between the prompt and each of the 5 options. Sort these options from highest similarity to lowest to form your top 3 predictions. What is the final average MAP@3 score of this TF-IDF pipeline across the entire training set?

In [16]:
matrix_scores = np.column_stack([sim_scores[opt] for opt in options])

# Top-3 predictions based on TF-IDF cosine similarity
top3_idx = np.argsort(matrix_scores, axis=1)[:, ::-1][:, :3]
top3_preds = np.array(options)[top3_idx]

# MAP@3
avg_map3 = np.mean([
    1/(list(pred).index(ans)+1) if ans in pred else 0
    for ans, pred in zip(df['answer'], top3_preds)
])

avg_map3

np.float64(0.25525)